In [1]:
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def extract_job_title(url):
    """
    Heuristic function to extract and clean the job title from the URL slug.
    """
    match = re.search(r'/jobs/detail/([^/]+)/', url)
    if not match:
        return "Unknown Role"
    
    slug = match.group(1)
    
   )
    slug = re.sub(r'%26%2347', '/', slug)
    slug = re.sub(r'%26%2345', '-', slug)
    slug = re.sub(r'%28|%29', '', slug)
    slug = re.sub(r'%2C', ',', slug)
    
    
    clean_title = slug.replace('-', ' ')
    
   
    words = clean_title.split()
    if len(words) > 3:
        clean_title = " ".join(words[:3])
        
    return clean_title.strip()

In [3]:
def build_and_run_recommender(csv_path, user_skills):
    """
    4-Stage Pipeline: Input -> Processing & Vectorization -> Scoring -> Output
    """
    print("Stage 1: Loading and Preprocessing Data...")
    df = pd.read_csv(csv_path)
    
    # Parse the stringified lists in 'raw_skills' into actual Python lists
    df['skills_list'] = df['raw_skills'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else []
    )
    
    # Convert lists into space-separated strings for TF-IDF analysis
    df['skills_text'] = df['skills_list'].apply(lambda x: ' '.join(x))
    
    # Extract job titles from the 'url' column
    df['job_title'] = df['url'].apply(extract_job_title)
    
    # Treat job roles as discrete items by grouping all skill mentions under that job title
    role_profiles = df.groupby('job_title')['skills_text'].apply(lambda x: ' '.join(x)).reset_index()
    
    print("Stage 2: Processing and Vectorization (TF-IDF)...")
    # Initialize TF-IDF Vectorizer
    vectorizer = TfidfVectorizer(stop_words='english')
    # Fit the vectorizer on our consolidated job role profiles
    tfidf_matrix = vectorizer.fit_transform(role_profiles['skills_text'])
    
    print("Stage 3: Scoring via Cosine Similarity...")
    # Clean user input to match formatting
    user_skills_text = ' '.join(user_skills).lower()
    
    # Map the user's skills to the established vector space
    user_vector = vectorizer.transform([user_skills_text])
    
    # Calculate the angular alignment (Cosine Similarity) against all role profiles
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix)
    
    print("Stage 4: Output Generation...\n")
    # Add scores back to the profiles dataframe
    role_profiles['similarity_score'] = similarity_scores[0]
    
  
    top_matches = role_profiles.sort_values(by='similarity_score', ascending=False).head(3)
    
    # Output Results
    print("=" * 50)
    print(f"INPUT ARRAY: {user_skills}")
    print("=" * 50)
    
    rank = 1
    for index, row in top_matches.iterrows():
        print(f"Rank {rank}: {row['job_title']}")
        print(f"Mathematical Alignment: {row['similarity_score']:.4f}")
        print("-" * 50)
        rank += 1

In [4]:
if __name__ == "__main__":

    dataset_path = "/Users/m12021/Downloads/raw_skills.csv"
    
   
    input_skills = ["Python", "Cloud", "Automation"]
    
    
    build_and_run_recommender(dataset_path, input_skills)

Stage 1: Loading and Preprocessing Data...
Stage 2: Processing and Vectorization (TF-IDF)...
Stage 3: Scoring via Cosine Similarity...
Stage 4: Output Generation...

INPUT ARRAY: ['Python', 'Cloud', 'Automation']
Rank 1: Enterprise Architect Level
Mathematical Alignment: 0.4415
--------------------------------------------------
Rank 2: IT Enterprise Architect
Mathematical Alignment: 0.4306
--------------------------------------------------
Rank 3: Enterprise Architect Encore
Mathematical Alignment: 0.3597
--------------------------------------------------
